# 🚀 XGBoost — Senjata Rahasia Pemenang Kompetisi ML

**Modul 1: Machine Learning Fundamentals** | Notebook 7 dari 9

---

## 🎯 Apa yang Akan Kita Pelajari?

1. Apa itu **XGBoost** dan mengapa ia begitu populer
2. Cara menggunakan XGBoost untuk **klasifikasi**
3. **Feature Importance** — fitur mana yang paling berpengaruh
4. **Early Stopping** — menghentikan training pada waktu yang tepat
5. Perbandingan XGBoost vs model lain

### ⏱️ Estimasi Waktu: ~90 menit + diskusi

---

## 💡 Apa Spesialnya XGBoost?

Di notebook sebelumnya, kita belajar **Gradient Boosting** — model yang belajar dari kesalahan secara berurutan. **XGBoost** (eXtreme Gradient Boosting) adalah versi **lebih cepat dan lebih pintar**.

### Mengapa XGBoost Menjadi Favorit?

| Keunggulan | Penjelasan |
|-----------|------------|
| **Cepat** | Bisa menggunakan semua CPU secara paralel |
| **Regularisasi** | Punya built-in pencegah overfitting |
| **Missing Values** | Bisa menangani data kosong secara otomatis |
| **Early Stopping** | Otomatis berhenti saat tidak ada peningkatan |

**Fakta menarik:** Antara 2015–2020, mayoritas pemenang kompetisi **Kaggle** (kompetisi data science terbesar dunia) menggunakan XGBoost untuk data tabular!

---

## 1️⃣ Persiapan

In [ ]:
# Install XGBoost
!pip install -q xgboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, classification_report
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('✅ Library berhasil di-import!')

---

## 2️⃣ Memuat Data — Income Prediction

Kita menggunakan dataset **Income Evaluation** — dataset dunia nyata dari sensus AS (Amerika Serikat).

**Pertanyaan:** Berdasarkan data demografis (umur, pendidikan, pekerjaan, dll), bisakah kita prediksi apakah seseorang berpenghasilan **di atas $50.000/tahun** atau tidak?

Dataset ini cukup besar (~32.000 baris) dan kompleks — cocok untuk menguji kekuatan XGBoost!

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Baca dan bersihkan dataset
df = pd.read_csv('income_evaluation.csv')

# Bersihkan nama kolom dan nilai (ada spasi di depan)
df.columns = [col.strip() for col in df.columns]
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# Hapus kolom fnlwgt (bobot sensus, tidak berguna untuk prediksi)
df = df.drop(columns='fnlwgt')

# Buat target numerik: >50K = 1, <=50K = 0
df['target'] = (df['income'] == '>50K').astype(int)

print(f'Dataset: {df.shape[0]:,} baris, {df.shape[1]} kolom')
print(f'\nTarget (Penghasilan):')
print(f'  ≤$50K: {(df["target"]==0).sum():,} ({(df["target"]==0).mean()*100:.1f}%)')
print(f'  >$50K: {(df["target"]==1).sum():,} ({(df["target"]==1).mean()*100:.1f}%)')

df.head()

### Persiapan Fitur

Kita akan menggunakan **fitur numerik** langsung, dan mengubah **fitur kategorikal** (teks) menjadi angka dengan One-Hot Encoding.

In [ ]:
# Pilih fitur (buang kolom income asli dan target)
fitur_kolom = [c for c in df.columns if c not in ['income', 'target']]

# One-Hot Encoding untuk fitur kategorikal
X = pd.get_dummies(df[fitur_kolom], drop_first=True)
y = df['target']

print(f'Jumlah fitur setelah encoding: {X.shape[1]}')
print(f'Jumlah data: {X.shape[0]:,}')

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nData latihan: {len(X_train):,} | Data ujian: {len(X_test):,}')

---

## 3️⃣ Melatih XGBoost

Kita menggunakan `XGBClassifier` — API yang sama seperti scikit-learn, jadi cara pakainya familiar!

### Parameter Penting XGBoost:

| Parameter | Fungsi | Analogi |
|-----------|--------|---------|
| `n_estimators` | Jumlah pohon | Berapa kali "belajar dari kesalahan" |
| `max_depth` | Kedalaman pohon | Seberapa detail tiap pelajaran |
| `learning_rate` | Kecepatan belajar | Langkah kecil = hati-hati, langkah besar = cepat tapi berisiko |
| `scale_pos_weight` | Bobot kelas minoritas | Memberi perhatian lebih pada kelas yang jarang |

In [ ]:
# Hitung rasio kelas untuk menangani data tidak seimbang
rasio = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Rasio kelas: {rasio:.1f}x lebih banyak ≤$50K dibanding >$50K')

# Latih XGBoost
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=rasio,   # Seimbangkan kelas
    random_state=42,
    n_jobs=-1,                # Gunakan semua CPU
    eval_metric='auc'         # Metrik evaluasi selama training
)

xgb_model.fit(X_train, y_train)
print('\n✅ XGBoost berhasil dilatih!')

In [ ]:
# Evaluasi
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print('📊 Hasil XGBoost')
print('=' * 40)
print(f'  Accuracy:  {acc:.3f} ({acc*100:.1f}%)')
print(f'  F1-Score:  {f1:.3f} ({f1*100:.1f}%)')
print(f'  AUC-ROC:   {auc:.3f} ({auc*100:.1f}%)')
print('=' * 40)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=['≤$50K', '>$50K']).plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix — XGBoost', fontsize=13, fontweight='bold')
plt.show()

---

## 4️⃣ Feature Importance — Fitur Mana yang Paling Berpengaruh?

Salah satu keunggulan XGBoost: bisa menunjukkan **fitur mana yang paling penting** untuk prediksi. Ini sangat berguna untuk memahami **mengapa** model membuat keputusan tertentu.

In [ ]:
# Feature Importance — Top 15 fitur terpenting
importance = pd.Series(xgb_model.feature_importances_, index=X.columns)
top_15 = importance.nlargest(15)

plt.figure(figsize=(10, 7))
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(top_15)))
top_15.plot(kind='barh', color=colors[::-1])
plt.xlabel('Importance Score', fontsize=12)
plt.title('Top 15 Fitur Terpenting — XGBoost', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Fitur-fitur terpenting untuk memprediksi penghasilan >$50K:')
for i, (fitur, score) in enumerate(top_15.items(), 1):
    if i <= 5:
        print(f'   {i}. {fitur} (skor: {score:.3f})')

---

## 5️⃣ Early Stopping — Berhenti di Waktu yang Tepat

Masalah: Jika kita melatih terlalu banyak pohon, model bisa **overfitting**. Tapi berapa jumlah yang pas?

**Early Stopping** = otomatis berhenti melatih ketika skor pada data validasi **tidak membaik** selama beberapa iterasi.

**Analogi:** Seperti belajar untuk ujian — ada saatnya belajar lebih lama justru membuat bingung (overlearning). Early stopping = tahu kapan harus berhenti belajar.

In [ ]:
# Bagi data train lagi untuk validasi
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# XGBoost dengan Early Stopping
xgb_early = XGBClassifier(
    n_estimators=1000,        # Maksimum 1000 pohon
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=rasio,
    random_state=42,
    n_jobs=-1,
    eval_metric='auc',
    early_stopping_rounds=20  # Berhenti jika 20 iterasi tidak membaik
)

xgb_early.fit(
    X_tr, y_tr,
    eval_set=[(X_tr, y_tr), (X_val, y_val)],
    verbose=50  # Tampilkan setiap 50 iterasi
)

print(f'\n🛑 Training berhenti di iterasi ke-{xgb_early.best_iteration}')
print(f'   (dari maksimum 1000 — menghemat {1000 - xgb_early.best_iteration} iterasi!)')

In [ ]:
# Visualisasi training progress
results = xgb_early.evals_result()

plt.figure(figsize=(10, 5))
plt.plot(results['validation_0']['auc'], label='Data Latihan', color='#2196F3', linewidth=1.5)
plt.plot(results['validation_1']['auc'], label='Data Validasi', color='#FF6F00', linewidth=1.5)
plt.axvline(x=xgb_early.best_iteration, color='green', linestyle='--', alpha=0.7,
            label=f'Best iteration ({xgb_early.best_iteration})')

plt.xlabel('Iterasi (Jumlah Pohon)', fontsize=12)
plt.ylabel('AUC Score', fontsize=12)
plt.title('Training Progress dengan Early Stopping', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('💡 Perhatikan: setelah titik hijau, skor validasi mulai TURUN')
print('   → training lebih lanjut justru membuat model LEBIH BURUK (overfitting)')

---

## 6️⃣ Perbandingan: XGBoost vs Random Forest vs Gradient Boosting

In [ ]:
# Latih model pembanding
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

gb_model = GradientBoostingClassifier(n_estimators=200, max_depth=6, random_state=42)
gb_model.fit(X_train, y_train)

# Bandingkan
models = {
    'Random Forest': rf_model,
    'Gradient Boosting': gb_model,
    'XGBoost': xgb_model,
}

print('📊 PERBANDINGAN MODEL')
print('=' * 60)
print(f'{"Model":<22} {"Accuracy":>10} {"F1-Score":>10} {"AUC-ROC":>10}')
print('-' * 60)

comparison = []
for nama, mdl in models.items():
    pred = mdl.predict(X_test)
    proba = mdl.predict_proba(X_test)[:, 1]
    acc_m = accuracy_score(y_test, pred)
    f1_m = f1_score(y_test, pred)
    auc_m = roc_auc_score(y_test, proba)
    comparison.append((nama, acc_m, f1_m, auc_m))
    print(f'  {nama:<20} {acc_m:>10.3f} {f1_m:>10.3f} {auc_m:>10.3f}')

print('=' * 60)

# Bar chart perbandingan
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics_names = ['Accuracy', 'F1-Score', 'AUC-ROC']
colors = ['#42A5F5', '#66BB6A', '#FF6F00']

for idx, metric in enumerate(metrics_names):
    values = [c[idx+1] for c in comparison]
    names = [c[0] for c in comparison]
    bars = axes[idx].bar(names, values, color=colors)
    axes[idx].set_title(metric, fontsize=13, fontweight='bold')
    axes[idx].set_ylim(min(values) - 0.05, max(values) + 0.02)
    for bar, v in zip(bars, values):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                       f'{v:.3f}', ha='center', fontsize=10)

plt.suptitle('Perbandingan Model Ensemble', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 📝 Ringkasan: Apa yang Kita Pelajari Hari Ini?

### Konsep Utama

1. **XGBoost** = versi super dari Gradient Boosting — lebih cepat, punya regularisasi built-in
2. **Feature Importance** menunjukkan fitur mana yang paling berpengaruh
3. **Early Stopping** otomatis menghentikan training saat mulai overfitting
4. `scale_pos_weight` menangani data tidak seimbang tanpa perlu SMOTE

### Kapan Menggunakan XGBoost?

✅ **Data tabular** (tabel/spreadsheet) — XGBoost adalah raja!
✅ Dataset **menengah hingga besar** (ribuan sampai jutaan baris)
✅ Kamu butuh **akurasi terbaik** dan punya waktu untuk tuning
✅ Kamu ingin tahu **fitur mana yang penting**

❌ Data **gambar, teks, suara** → gunakan Deep Learning
❌ Dataset **sangat kecil** (<100 baris) → model sederhana cukup

### Fun Fact: XGBoost vs Deep Learning

| Jenis Data | Pemenang |
|-----------|----------|
| Tabel/spreadsheet | **XGBoost** (hampir selalu!) |
| Gambar | Deep Learning (CNN) |
| Teks | Deep Learning (Transformer/LLM) |
| Audio | Deep Learning |

---

### 🔜 Notebook Selanjutnya: K-Means & Hierarchical Clustering
Sejauh ini kita belajar **supervised learning** (ada label/jawaban). Selanjutnya: **unsupervised learning** — bagaimana komputer menemukan kelompok-kelompok dalam data **tanpa** dikasih tahu jawabannya!